# 🧾 生成式AI應用開發｜第06週
## Structured Outputs、JSON Schema 與資料抽取

> **執行環境：Google Colab** ｜ **主線：OpenAI Responses API**

本週把第 4 週 Prompt Engineering 和第 5 週 API 狀態管理，推進到「可被程式穩定處理」的結構化輸出。核心目標不是讓模型回答得漂亮，而是讓模型輸出能直接變成 Python dict、資料表、表單欄位或後續 App 的輸入。

官方文件入口：
- Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs
- Responses API: https://platform.openai.com/docs/api-reference/responses

> **本週任務**
> 使用 JSON Schema 約束模型輸出，完成課程事件、作業清單與客服資料抽取。


> **學生版**：保留課堂練習 TODO。請先完成練習 A，再依時間完成練習 B 與選做練習 C。


## 🎯 本週學習目標

| # | 能力 | 後續應用 |
|---|------|----------|
| 1 | 分辨一般 JSON prompt 與 Structured Outputs | 降低輸出格式漂移 |
| 2 | 撰寫基本 JSON Schema | 表單、資料抽取、資料庫欄位 |
| 3 | 使用 Responses API 的 `text.format` | 讓模型遵守指定 schema |
| 4 | 將 `response.output_text` 轉成 Python dict | 串接 App 邏輯 |
| 5 | 驗證必填欄位與型別 | 防止錯誤資料進入系統 |
| 6 | 批次測試資料抽取結果 | 建立可維護的 Prompt / Schema |
| 7 | 設計自己的抽取器 | 期中、期末專題資料處理 |


## 🗺️ 三堂課（150 分鐘）實作流程

| 時間 | 主題 | 產出 |
|------|------|------|
| 0-15 分 | 非結構化輸出的問題 | 看懂為什麼只靠 prompt 不夠 |
| 15-35 分 | JSON Schema 基礎 | 寫出第一個 event schema |
| 35-60 分 | Structured Outputs API | 取得可解析的 JSON 字串 |
| 60-80 分 | Python 驗證與錯誤處理 | 建立 `validate_required_fields()` |
| 80-105 分 | 陣列與巢狀資料抽取 | 作業清單抽取器 |
| 105-130 分 | 批次測試與 schema 版本 | 比較多筆輸入是否穩定 |
| 130-150 分 | 三組課堂練習 | 完成自己的抽取器 |


---
## 0. 環境準備

本週沿用第 3-5 週的 OpenAI API 設定方式。請先確認 Colab Secrets 或本機環境變數中已有 `OPENAI_API_KEY`。


In [ ]:
# 安裝或更新 OpenAI Python SDK
# Pydantic 用於進階 structured parsing 與欄位驗證。
!pip install -q --upgrade openai pydantic


In [ ]:
import json
import os
from copy import deepcopy
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則改用既有環境變數。
try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("找不到 OPENAI_API_KEY。請先設定 Colab Secret 或本機環境變數。")

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI()
print("本週模型：", MODEL)
print("API key 已設定（不顯示內容）")


In [ ]:
STRUCTURED_INSTRUCTIONS = """
你是資料抽取助理。請只根據使用者提供的文字抽取資料。
如果資料沒有出現，請使用 null 或空陣列，不要自行補充。
"""


def structured_response(user_input, schema, name="structured_output", instructions=STRUCTURED_INSTRUCTIONS):
    """使用 Responses API 與 JSON Schema 取得結構化輸出。"""
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")
    if not isinstance(schema, dict) or schema.get("type") != "object":
        raise ValueError("schema 必須是 JSON Schema object。")

    response = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=user_input,
        text={
            "format": {
                "type": "json_schema",
                "name": name,
                "schema": schema,
                "strict": True,
            }
        },
    )
    return response


def _read_attr_or_key(obj, name, default=None):
    """同時支援 SDK 物件與 dict，方便檢查回應內容。"""
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def get_refusal_reason(response):
    """Structured Outputs 遇到安全拒答時，回應可能不符合原 schema。"""
    for output_item in _read_attr_or_key(response, "output", []) or []:
        for content_item in _read_attr_or_key(output_item, "content", []) or []:
            refusal = _read_attr_or_key(content_item, "refusal")
            if refusal:
                return refusal
    return None


def parse_output_text(response):
    """將 Structured Outputs 的 JSON 字串轉成 Python dict。"""
    refusal = get_refusal_reason(response)
    if refusal:
        raise ValueError(f"模型拒答，未產生符合 schema 的資料：{refusal}")
    output_text = getattr(response, "output_text", "")
    if not output_text:
        raise ValueError("回應沒有 output_text，請檢查 API 回應或模型是否拒答。")
    return json.loads(output_text)


def print_json(data, title=None):
    """格式化顯示 dict/list，方便課堂觀察。"""
    if title:
        print(f"=== {title} ===")
    print(json.dumps(data, ensure_ascii=False, indent=2))

> **成本提醒**
> 本週多數 API cell 都會產生付費請求。批次測試與延伸實驗預設關閉，確認額度後再把 `RUN_...` 變數改為 `True`。


---
## 1. 為什麼不能只叫模型「請輸出 JSON」？

Prompt 可以要求模型輸出 JSON，但模型仍可能多加說明文字、漏掉欄位、改欄位名稱，或把數字輸出成文字。Structured Outputs 的目標，是讓輸出符合你提供的 JSON Schema。


In [ ]:
loose_prompt = """
請把以下文字整理成 JSON：
下週三 14:00 在 A203 教室有 AI 專題討論，負責人是小安。
"""

RUN_LOOSE_JSON_DEMO = False
if RUN_LOOSE_JSON_DEMO:
    loose = client.responses.create(
        model=MODEL,
        instructions="你是資料整理助理，請用繁體中文回答。",
        input=loose_prompt,
    )
    print(loose.output_text)
else:
    print("一般 JSON prompt 示範尚未執行；確認成本後可將 RUN_LOOSE_JSON_DEMO 改為 True。")


### ✍️ 觀察紀錄

執行上方非結構化版本時，請觀察：

- 是否有多餘說明文字？
- 欄位名稱是否固定？
- 日期、時間、地點是否容易被穩定取出？
- 如果要交給程式處理，還需要做哪些清理？


---
## 2. JSON Schema 的基本組成

| Schema 欄位 | 用途 |
|-------------|------|
| `type` | 指定資料型態，例如 `object`、`string`、`array` |
| `properties` | 定義 object 內有哪些欄位 |
| `required` | 指定一定要出現的欄位 |
| `additionalProperties` | 是否允許未列出的欄位 |

在 `strict=True` 時，建議清楚列出 `required`，並設定 `additionalProperties: False`。


In [ ]:
event_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "事件名稱"},
        "date": {"type": ["string", "null"], "description": "日期，保留原文格式"},
        "time": {"type": ["string", "null"], "description": "時間，保留原文格式"},
        "location": {"type": ["string", "null"], "description": "地點"},
        "owner": {"type": ["string", "null"], "description": "負責人"},
    },
    "required": ["title", "date", "time", "location", "owner"],
    "additionalProperties": False,
}

print_json(event_schema, title="event_schema")


### 🧱 Schema 設計原則

欄位名稱要清楚、穩定，並接近程式會使用的資料名稱。欄位描述不是給學生看的註解，而是提供模型判斷欄位意義的線索。


In [ ]:
event_text = "下週三 14:00 在 A203 教室有 AI 專題討論，負責人是小安。"

RUN_EVENT_EXTRACTION = False
if RUN_EVENT_EXTRACTION:
    event_response = structured_response(
        event_text,
        event_schema,
        name="course_event",
        instructions="你是課程行政資料抽取助理。請抽取事件資訊。",
    )
    event_data = parse_output_text(event_response)
    print_json(event_data, title="抽取結果")
else:
    print("事件抽取尚未執行；確認 API 成本後可將 RUN_EVENT_EXTRACTION 改為 True。")


### 🔍 輸出解析心智模型

Structured Outputs 讓 `response.output_text` 更容易是符合 schema 的 JSON 字串；Python 仍需用 `json.loads()` 轉成 dict，後續才能傳給表單、資料表或資料庫。


---
## 3. 程式端驗證：不要只相信模型

Schema 能降低格式錯誤，但正式 App 仍應在程式端檢查必填欄位、重要欄位是否為空，以及是否有額外欄位進入系統。


In [ ]:
def validate_required_fields(data, required_fields):
    """檢查必填欄位是否存在；回傳錯誤訊息 list。"""
    errors = []
    for field in required_fields:
        if field not in data:
            errors.append(f"缺少欄位：{field}")
    return errors


def validate_no_extra_fields(data, allowed_fields):
    """檢查是否有 schema 未允許的欄位。"""
    extras = sorted(set(data) - set(allowed_fields))
    return [f"不允許的欄位：{field}" for field in extras]


def validate_event_data(data):
    """針對 event_schema 做最基本的程式端驗證。"""
    required = event_schema["required"]
    allowed = event_schema["properties"].keys()
    return validate_required_fields(data, required) + validate_no_extra_fields(data, allowed)

sample_event = {"title": "AI 專題討論", "date": "下週三", "time": "14:00", "location": "A203 教室", "owner": "小安"}
print(validate_event_data(sample_event))


### ⚖️ Schema 驗證與商業規則驗證

JSON Schema 負責「格式」；你的程式仍要負責「商業規則」。例如日期是否真的存在、教室是否可借用、負責人是否為本班學生，這些通常需要額外資料來源。


---
## 4. 陣列與巢狀資料：抽取多筆作業


In [ ]:
assignment_schema = {
    "type": "object",
    "properties": {
        "assignments": {
            "type": "array",
            "description": "文字中提到的作業清單",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "due_date": {"type": ["string", "null"]},
                    "submission": {"type": ["string", "null"]},
                    "late_policy": {"type": ["string", "null"]},
                },
                "required": ["name", "due_date", "submission", "late_policy"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["assignments"],
    "additionalProperties": False,
}

assignment_text = """
作業一是 Prompt 改寫練習，9/30 前上傳 Moodle。
作業二是聊天機器人原型，10/15 23:59 前繳交 GitHub 連結，逾期不收 email 補交。
"""

print_json(assignment_schema, title="assignment_schema")


In [ ]:
RUN_ASSIGNMENT_EXTRACTION = False
if RUN_ASSIGNMENT_EXTRACTION:
    assignment_response = structured_response(
        assignment_text,
        assignment_schema,
        name="course_assignments",
        instructions="你是課程作業資訊抽取助理。請只抽取文字明確提供的資訊。",
    )
    assignment_data = parse_output_text(assignment_response)
    print_json(assignment_data, title="作業抽取結果")
else:
    print("作業清單抽取尚未執行；確認 API 成本後可將 RUN_ASSIGNMENT_EXTRACTION 改為 True。")


### 🧩 陣列 Schema 的常見錯誤

- 忘記在 `items` 裡定義每一筆資料的 object 結構
- 忘記每層 object 都設定 `required`
- 允許太多額外欄位，導致後續程式難以處理
- 欄位名稱使用中文或空白，造成程式端不方便存取


---
## 5. 安全解析與錯誤處理

正式 App 不應假設 API 永遠成功，也不應假設 `output_text` 永遠能被 `json.loads()` 解析。


In [ ]:
def extract_structured_safe(user_input, schema, name, instructions=STRUCTURED_INSTRUCTIONS):
    """安全版抽取：成功回傳 (True, data)，失敗回傳 (False, error_message)。"""
    try:
        response = structured_response(user_input=user_input, schema=schema, name=name, instructions=instructions)
        data = parse_output_text(response)
        return True, data
    except json.JSONDecodeError as exc:
        return False, f"JSON 解析失敗：{exc}"
    except Exception as exc:
        return False, f"Structured Outputs 呼叫失敗：{exc}"

RUN_SAFE_EXTRACT_DEMO = False
if RUN_SAFE_EXTRACT_DEMO:
    ok, result = extract_structured_safe(event_text, event_schema, "course_event")
    print_json(result if ok else {"error": result})
else:
    print("安全抽取示範尚未執行；確認 API 成本後可將 RUN_SAFE_EXTRACT_DEMO 改為 True。")


### 🛑 無法完成時要怎麼處理？

Structured Outputs 仍可能遇到 API 錯誤、輸出被中斷、模型拒答或資料不足。正式應用至少要有友善錯誤訊息、保留原始輸入以便重試，並避免把不完整資料寫入資料庫。

特別注意：安全拒答不一定會符合你指定的 schema。程式端要先檢查 refusal 或空的 `output_text`，再做 `json.loads()`，不要假設每一次回應都能直接寫入資料庫。

---
## 6. 批次測試：確認 Schema 是否穩定

單筆成功不代表 Schema 穩定。至少要準備幾筆不同格式的文字，觀察欄位是否一致。


In [ ]:
event_test_cases = [
    "本週五 10:00 在 B101 開會，主題是期中專題檢查，負責人小華。",
    "明天下午到圖書館討論 Streamlit 部署，沒有指定負責人。",
    "10/20 晚上 7 點線上舉辦 GitHub 操作複習，由助教阿明帶領。",
]


def run_event_batch_tests(cases):
    results = []
    for index, text in enumerate(cases, start=1):
        ok, data = extract_structured_safe(text, event_schema, name="course_event", instructions="你是課程活動資料抽取助理。")
        errors = [] if not ok else validate_event_data(data)
        results.append({"case": index, "ok": ok and not errors, "data": data, "errors": errors})
    return results

RUN_EVENT_BATCH_TESTS = False
if RUN_EVENT_BATCH_TESTS:
    print_json(run_event_batch_tests(event_test_cases), title="批次測試結果")
else:
    print("批次測試尚未執行；確認 API 成本後可將 RUN_EVENT_BATCH_TESTS 改為 True。")


### 🔁 Schema 改版紀錄與進階路線

當你發現抽取結果不穩定時，不要只改 prompt。請同時記錄 schema 版本、改了什麼，以及為什麼。

本週主線先使用手寫 JSON Schema，因為它最能看懂 Structured Outputs 的底層結構。接著補充 Pydantic 版本，讓後續 Python App 可以用類別定義資料格式，並用 `client.responses.parse()` 直接取得已驗證的 Python 物件。


In [ ]:
schema_registry = {
    "course_event_v1": deepcopy(event_schema),
    "course_assignments_v1": deepcopy(assignment_schema),
}


def list_schema_names(registry):
    return sorted(registry.keys())


def get_schema(registry, name):
    if name not in registry:
        raise KeyError(f"找不到 schema：{name}")
    return deepcopy(registry[name])


print("目前 schema：", list_schema_names(schema_registry))


---
## 7. Pydantic 進階：用類別定義輸出格式

手寫 JSON Schema 適合理解底層結構；Pydantic 適合 Python 專案維護資料格式。它可以把欄位型別、值域限制與巢狀物件寫成類別，並搭配 `client.responses.parse()` 直接取得 `response.output_parsed`。

| 技巧 | 用途 |
|------|------|
| `Optional[str] = None` | 欄位可能不存在 |
| `Literal["a", "b"]` | 限制只能是固定幾種值，類似 enum |
| `Field(ge=1, le=5)` | 數字要落在指定範圍 |
| `list[Model]` | 一個欄位是另一個模型的清單 |
| `ValidationError` | 捕捉欄位型別或限制不符合的狀況 |

> 若課堂環境的 SDK 尚未支援 `responses.parse()`，請先回到前面的 `text.format` + `json.loads()` 主線。

In [ ]:
# Pydantic 範例：評論資料抽取
review_text = "王小明給了這家餐廳五顆星，他說：食物非常好吃，服務也很親切，會再來！"


class ReviewInsight(BaseModel):
    customer_name: str
    # TODO 1：新增 rating 欄位，型別 int，限制範圍 1~5（提示：Field(ge=1, le=5)）
    # TODO 2：新增 sentiment 欄位，只能是 "positive" / "neutral" / "negative"（提示：Literal）
    # TODO 3：新增 summary 欄位，型別 str，代表一句話摘要


class ActionItem(BaseModel):
    owner: str
    task: str
    due_date: Optional[str] = None


class MeetingMinutes(BaseModel):
    meeting_topic: str
    attendees: list[str]
    action_items: list[ActionItem]


def parse_with_pydantic(text, model_cls, instructions="你是嚴謹的資料抽取助理，只依照提供文字填寫欄位。"):
    """使用 responses.parse() 直接取得 Pydantic 物件。"""
    response = client.responses.parse(
        model=MODEL,
        instructions=instructions,
        input=text,
        text_format=model_cls,
    )
    return response.output_parsed


def extract_structured(text, model_cls, instructions="你是嚴謹的資料抽取助理，只依照提供文字填寫欄位，不要編造。"):
    """App 可重用的 Pydantic 抽取 wrapper；成功回傳物件，失敗回傳 None。"""
    try:
        return parse_with_pydantic(text, model_cls, instructions=instructions)
    except ValidationError as exc:
        print("欄位驗證失敗：", exc)
        return None
    except Exception as exc:
        print("Structured parsing 失敗：", exc)
        return None


meeting_text = """
今天的產品會議由小美主持，出席者有小美、志明、雅婷。
決議事項：
1. 志明要在下週五前完成新版首頁設計。
2. 雅婷負責整理使用者訪談紀錄，沒有明確期限。
3. 小美會發送會議記錄給全體團隊，今天之內完成。
"""

RUN_PYDANTIC_REVIEW_DEMO = False
if RUN_PYDANTIC_REVIEW_DEMO:
    insight = extract_structured(
        "請從以下評論抽取顧客姓名、評分、情緒與一句話摘要：\n\n" + review_text,
        ReviewInsight,
    )
    print(insight)
else:
    print("Pydantic review 示範尚未執行；完成 ReviewInsight 後再改為 True。")

RUN_MEETING_MINUTES_DEMO = False
if RUN_MEETING_MINUTES_DEMO:
    minutes = extract_structured(
        "請從以下會議記錄抽取會議主題、出席者與待辦事項：\n\n" + meeting_text,
        MeetingMinutes,
    )
    print(minutes)
else:
    print("巢狀 MeetingMinutes 示範尚未執行；確認 API 成本後可改為 True。")

---
## 8. 課堂練習 A：客服訊息資料抽取器

請用手寫 JSON Schema 完成客服訊息抽取器。這題延續前半段主線，重點是練習 enum、boolean 與必要欄位。


In [ ]:
# 練習 A：客服訊息資料抽取器
support_schema = {
    "type": "object",
    "properties": {
        # TODO 1：定義 category，建議 enum: 功能問題、帳務問題、操作疑問、其他
        # TODO 2：定義 urgency，建議 enum: low、medium、high
        # TODO 3：定義 summary，型別為 string
        # TODO 4：定義 needs_reply，型別為 boolean
    },
    # TODO 5：列出 required 欄位
    # TODO 6：設定 additionalProperties 為 False
}

support_message = "我昨天被扣款兩次，後台訂閱狀態還顯示未啟用，請今天幫我處理。"

# TODO 7：呼叫 extract_structured_safe()，印出抽取結果


---
### 練習 B：收據資料抽取器

請用 Pydantic 定義訂單資料格式，抽出訂單編號、顧客、商品清單與運費。這題展示結構化資料抽取後，可以直接進行小計與總額計算。


In [ ]:
receipt_text = """
訂單編號 A1023，顧客：陳建宏。
購買項目：
- 藍牙耳機 x2，單價 990 元
- 保護殼 x1，單價 250 元
運費 60 元。
"""


class OrderItem(BaseModel):
    item_name: str
    # TODO 1：新增 quantity，型別 int，限制至少 1
    # TODO 2：新增 unit_price，型別 int，限制至少 0


class OrderInfo(BaseModel):
    order_id: str
    # TODO 3：新增 customer_name，型別 str
    # TODO 4：新增 items，型別 list[OrderItem]
    # TODO 5：新增 shipping_fee，型別 int，限制至少 0


# TODO 6：使用 parse_with_pydantic(receipt_text, OrderInfo) 抽取資料
# TODO 7：若成功，計算商品小計與加運費後總額


In [ ]:
def extract_with_retry(text, model_cls, retries=2):
    """選做：只針對暫時性失敗重試；不要用 retry 掩蓋 schema 設計錯誤。"""
    # TODO 1：用迴圈呼叫 parse_with_pydantic(text, model_cls)，最多嘗試 retries + 1 次
    # TODO 2：捕捉 ValidationError 與一般 Exception，印出第幾次失敗
    # TODO 3：只要有一次成功就直接回傳結果
    # TODO 4：全部失敗時回傳 None
    pass


# TODO 5：用 ReviewInsight 或 OrderInfo 測試 extract_with_retry()


---
## ✅ 完成檢核

- [ ] 能說明 Structured Outputs 與一般 JSON prompt 的差異
- [ ] 能寫出包含 `type`、`properties`、`required`、`additionalProperties` 的 schema
- [ ] 能用 `text.format` 呼叫 Structured Outputs
- [ ] 能將 `response.output_text` 解析成 Python dict
- [ ] 能處理模型拒答、空輸出與 JSON 解析失敗
- [ ] 能做基本必填欄位與額外欄位檢查
- [ ] 能用 Pydantic 定義欄位限制與巢狀資料
- [ ] 能完成至少一個自己的資料抽取器

## 📝 課後任務與延伸閱讀

請選擇你的期中專題，設計一個資料抽取器：找一段真實或模擬輸入、設計 JSON Schema、用 Structured Outputs 抽取資料、做至少三筆測試，並記錄 schema v1 到 v2 的修改理由。

延伸閱讀：
- OpenAI Structured Outputs: https://platform.openai.com/docs/guides/structured-outputs
- Responses API reference: https://platform.openai.com/docs/api-reference/responses
- JSON Schema: https://json-schema.org/learn/getting-started-step-by-step

下週會銜接 Streamlit 與部署流程，把本週抽取出的 dict 放進可操作的網頁介面。
